# Proyecto Final - Machine Learning
Dataset: Evaluacion de Riesgo Crediticio (German Credit)

En este proyecto aplico lo aprendido durante las clases para resolver un problema real de scoring crediticio: predecir si un solicitante de prestamo sera buen pagador o caera en mora (default), justificando cada paso metodologico desde el preprocesamiento hasta modelos supervisados y no supervisados.

## 1. Carga de datos y exploracion inicial (Relacionado con Sesion 1)
Como vimos en la sesion 1 sobre los tipos de aprendizaje, este es un problema de aprendizaje supervisado de clasificacion binaria porque contamos con una variable objetivo etiquetada (default): 0 para clientes que pagan a tiempo y 1 para clientes morosos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/credit.csv"
df = pd.read_csv(url)

# Mapeamos la variable objetivo: 1 a 0 (buen pagador) y 2 a 1 (moroso)
df["default"] = df["default"].map({1: 0, 2: 1})

print("Forma del dataset:", df.shape)
print("Distribucion de la variable objetivo:")
print(df["default"].value_counts(normalize=True))
df.head(3)


## 2. Particion y construccion del Pipeline (Relacionado con Sesion 2 y 3)
En la sesion 2 vimos que para evitar fuga de datos (data leakage) debemos separar los datos antes de cualquier transformacion. Y en la sesion 3 vimos el uso de Pipelines y ColumnTransformer.
La decision aqui fue hacer un train/test split (70% train y 30% test) estratificado porque la clase morosa representa el 30% del dataset. Despues armamos un preprocesador que estandariza variables numericas con StandardScaler y aplica OneHotEncoder a las categoricas con handle_unknown='ignore' para soportar categorias nuevas en produccion.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop("default", axis=1)
y = df["default"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)

print(f"Variables numericas: {len(num_cols)} | Categoricas: {len(cat_cols)}")
print(f"Muestras train: {X_train.shape[0]} | Muestras test: {X_test.shape[0]}")


## 3. Entrenamiento y comparacion de clasificadores (Relacionado con Sesion 4)
Siguiendo lo visto en la sesion 4 sobre clasificadores supervisados, decidi comparar cuatro familias de modelos:
- Regresion Logistica (modelo lineal probabilistico con sigmoide)
- Arbol de Decision (modelo basado en divisiones no lineales)
- SVM con kernel RBF (modelo basado en vectores de soporte)
- Random Forest (ensamble para reducir varianza)
Cada modelo se encapsula en su Pipeline para que el escalado y codificacion se ajusten solo con los datos de entrenamiento.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "Regresion Logistica": LogisticRegression(max_iter=1000, random_state=42),
    "Arbol Decision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "SVM (RBF)": SVC(probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

trained_pipes = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe

print("Modelos entrenados correctamente dentro de sus respectivos pipelines.")


## 4. Evaluacion de metricas y matrices de confusion (Relacionado con Sesion 5)
En la sesion 5 aprendimos que no debemos guiarnos unicamente por el Accuracy cuando hay clases desbalanceadas. En riesgo crediticio, el error mas costoso para el banco es prestarle a alguien que no pagara (falso negativo). Por lo tanto, ademas del Accuracy evalue el Recall de morosos y el ROC-AUC, apoyandome en las matrices de confusion.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

results = []
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, (name, pipe) in enumerate(trained_pipes.items()):
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    rec = recall_score(y_test, preds)
    prec = precision_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    results.append({
        "Modelo": name,
        "Accuracy": acc,
        "Recall": rec,
        "Precision": prec,
        "F1": f1,
        "ROC-AUC": auc
    })
    
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i],
                xticklabels=["Pagador", "Moroso"], yticklabels=["Pagador", "Moroso"])
    axes[i].set_title(name)
    axes[i].set_xlabel("Prediccion")
    axes[i].set_ylabel("Real")

plt.tight_layout()
plt.show()

df_metrics = pd.DataFrame(results)
print(df_metrics.to_string(index=False))


## 5. Optimizacion de hiperparametros (Relacionado con Sesion 6)
Como la Regresion Logistica obtuvo el mejor balance general con mayor Recall (0.533) y ROC-AUC (0.801), procedi a optimizar sus hiperparametros.
En la sesion 6 aprendimos a usar GridSearchCV y StratifiedKFold para afinar la regularizacion (parametro C) mediante validacion cruzada de 5 particiones, evitando el sobreajuste y buscando la maxima generalizacion.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 5.0, 10.0],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"]
}

grid_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(grid_pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)[:, 1]

print("Mejores parametros encontrados:", grid.best_params_)
print(f"ROC-AUC en validacion cruzada: {grid.best_score_:.4f}")
print(f"ROC-AUC en test: {roc_auc_score(y_test, test_probs):.4f}")
print(f"Recall en test: {recall_score(y_test, test_preds):.4f}")


## 6. Analisis No Supervisado con PCA y K-Means (Relacionado con Sesiones 7 y 8)
En las sesiones 7 y 8 vimos clustering y reduccion de dimensionalidad. Decidi aplicar PCA para comprimir todas las variables procesadas a 2 componentes principales y despues usar K-Means con K=3 para identificar si los clientes se agrupan de forma natural por sus caracteristicas socioeconomicas sin conocer de antemano su historial de pago.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

X_all_prep = preprocessor.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all_prep)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_all_prep)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Visualizacion de clusters KMeans
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=cluster_labels, palette="Set1", ax=axes[0], alpha=0.8)
axes[0].set_title("Segmentacion K-Means (K=3) en espacio PCA")
axes[0].set_xlabel("Componente Principal 1")
axes[0].set_ylabel("Componente Principal 2")

# Visualizacion de morosidad real
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette={0: "blue", 1: "red"}, ax=axes[1], alpha=0.7)
axes[1].set_title("Distribucion real de impago (Rojo: Moroso, Azul: Pagador)")
axes[1].set_xlabel("Componente Principal 1")
axes[1].set_ylabel("Componente Principal 2")

plt.tight_layout()
plt.show()

print(f"Varianza explicada por los 2 componentes: {pca.explained_variance_ratio_.sum()*100:.2f}%")


## 7. Justificacion de decisiones y conclusiones
Durante el desarrollo del proyecto se tomaron decisiones tecnicas acordes a lo visto en clases:
- Particion y preprocesamiento en Pipeline: Se dividio train y test antes de cualquier ajuste y se uso ColumnTransformer para evitar data leakage entre conjuntos (sesion 2 y 3).
- Criterio de evaluacion: Debido al desbalance se priorizo el Recall y el ROC-AUC en lugar del Accuracy, ya que los falsos negativos representan un costo financiero directo para la institucion (sesion 5).
- Modelo seleccionado: La Regresion Logistica calibrada con GridSearchCV ofrecio el mejor desempeño y estabilidad sin sobreajustarse (sesion 4 y 6).
- Exploracion no supervisada: El uso conjunto de PCA y K-Means permitio visualizar y segmentar perfiles de clientes en tres grupos naturales, lo cual sirve para ofrecer condiciones de credito personalizadas (sesion 7 y 8).